# ✅ BIST KANITLI SİNYAL — GÜN İÇİ (intraday) sürüm

Bu notebook, günlük (daily) veride backtest'i geçen **5 kanıtlı sinyalin** (BANKER 🏆,
ICHIMOKU, ENGULFING, DERİN_DEĞER, CMI_KOMBO) **birebir aynı tanımlarını** ve **aynı çıkış
bracket'ini** (STOP=1ATR · +1.5R yarı+başabaş · +3R · 10 bar) gün içi bar'lara taşır.

## ⚠️ Dürüstlük notu — bunu atlamadan oku

Orijinal edge'ler (**BANKER +0.264R**, ICHIMOKU +0.066R, ENGULFING +0.050R, DERİN_DEĞER
+0.046R, CMI_KOMBO +0.033R) **günlük** bar'larda ölçüldü. `in_daily` → `in_1_hour` yapıp
"hâlâ kanıtlı" demek **yanlış** olur: 52-hafta aralığı, EMA200, "10 gün" süre stopu gün
içinde farklı bir istatistiksel rejimdir. Bu notebook'un kendi disiplin kuralı da nettir:

> *"Değişiklik yapma: bir fikri ancak canlı defter backtest'ten saparsa VE yeni backtest
> doğrularsa uygula."*

Bu yüzden bu sürüm iki şey yapar:
1. **Sinyalleri birebir korur** — hiçbir kural değişmedi, elenen kurallar geri eklenmedi,
   yfinance yok (tvDatafeed).
2. **Gün içi verisiyle edge'i YENİDEN DOĞRULAR** (Hücre 9): aynı 5 sinyal + aynı çıkış,
   gün içi bar'larda, rastgele girişe karşı `excess_R` ve `t` ile ölçülür. **Sadece bu
   testte `excess_R>0` ve `t≳2` çıkan sinyallere gün içinde güven.**

> **📌 1h koşusunun sonucu (Hücre 10 sonrası bölümde ayrıntılı):** günlük edge gün içine
> temiz transfer OLMADI. **ENGULFING** sağlam kaldı (t=3.51); **DERIN_DEGER zararlı** (t=−6.11);
> **BANKER hiç tetiklenmedi** (0 işlem) → BANKER-ağırlıklı konviksiyon gün içinde işlevsiz.
> Strateji kodu bilinçli değiştirilmedi; canlıya sokmadan önce aşağıdaki bulgu notunu oku.

## Zaman dilimi seçimi
Varsayılan **1 saat (1h)**: ~8 bar/gün → 5000 bar ≈ 2.5 yıl geçmiş (doğrulamaya yeter),
EMA200 (≈5 hafta) ve 252-bar aralık (≈1.5 ay) gün içinde anlamlı kalır. `CFG["INTERVAL"]`
ile `5m/15m/30m/1h/2h` arasında değiştirilebilir; tüm pencere/çıkış mekaniği bar-tabanlı
olduğu için kod aynen çalışır — **ama her yeni zaman dilimini Hücre 9 ile yeniden doğrula.**

> **Bu bir yatırım tavsiyesi değildir.** Gün içi edge daha küçük ve komisyon/kayma etkisi
> daha yüksektir; disiplin ve canlı defterle doğrulama şarttır.

In [ ]:
# === HÜCRE 1: KURULUM ===
import subprocess,sys
def _pip(*p): subprocess.run([sys.executable,"-m","pip","install","-q",*p],check=False)
_pip("--upgrade","git+https://github.com/rongardF/tvdatafeed.git")
_pip("tradingview-screener","pandas","numpy","matplotlib","openpyxl")
import os,gc,warnings,numpy as np,pandas as pd,matplotlib.pyplot as plt
warnings.filterwarnings("ignore"); from datetime import datetime
plt.rcParams.update({"figure.facecolor":"#0d1117","axes.facecolor":"#0d1117","savefig.facecolor":"#0d1117",
  "text.color":"#e6edf3","axes.labelcolor":"#e6edf3","xtick.color":"#8b949e","ytick.color":"#8b949e",
  "axes.edgecolor":"#30363d","grid.color":"#21262d"})
try:
    from google.colab import drive; drive.mount("/content/drive"); BASE="/content/drive/MyDrive/BIST_KanitliSinyal_Gunici"
except Exception: BASE="./BIST_KanitliSinyal_Gunici"
os.makedirs(BASE,exist_ok=True); print("Klasör:",BASE)
def sf(x,f="{:.2f}"):
    try: return f.format(x) if x is not None and np.isfinite(x) else "—"
    except Exception: return "—"

In [ ]:
# === HÜCRE 2: AYARLAR — GÜN İÇİ (çıkış mekaniği backtest'te DOĞRULANMIŞ bracket) ===
from tvDatafeed import TvDatafeed, Interval

# --- Zaman dilimi (gün içi). Seçenekler: '5m','15m','30m','1h','2h' ---
# Varsayılan '1h': ~8 bar/gün → 5000 bar ≈ 2.5 yıl (edge'i yeniden doğrulamaya yeter).
INTERVAL_STR = "1h"
_TV_INT = {"5m":Interval.in_5_minute,"15m":Interval.in_15_minute,"30m":Interval.in_30_minute,
           "1h":Interval.in_1_hour,"2h":Interval.in_2_hour}
_BPD    = {"5m":96,"15m":32,"30m":16,"1h":8,"2h":4}   # BIST seansı ~8s → yaklaşık bar/gün
TV_INTERVAL  = _TV_INT[INTERVAL_STR]
BARS_PER_DAY = _BPD[INTERVAL_STR]

CFG = dict(
  INTERVAL=INTERVAL_STR, BARS_PER_DAY=BARS_PER_DAY,
  N_BARS_TARAMA=800,       # canlı tarama: göstergeler için yeterli, hızlı
  N_BARS_BACKTEST=5000,    # yeniden doğrulama: maksimum geçmiş
  MIN_FIYAT=1.0, MIN_LIKIT_MTL=20, UNIVERSE_LIMIT=2000,
  TARAMA_SEMBOL=0,         # canlı taramada kaç hisse (0 = TÜM evren ~600). Hızlandırmak istersen örn. 200 (en likit) yaz.
  # Çıkış bracket'i — günlükte +0.264R…+0.033R veren mekanik; gün içi bar'a AYNEN uygulanır:
  ATR_LEN=14, ATR_STOP=1.0, T1_R=1.5, T2_R=3.0, SURE=10,   # DİKKAT: SURE = bar sayısı (gün değil!)
  GATE=60.0,               # teknik ucuzluk eşiği (DERİN_DEĞER/BANKER)
  SERMAYE=100_000, RISK_PCT=0.01, MAX_AL=15,
  # Yeniden doğrulama backtest'i:
  BT_SEMBOL=80,            # en likit N hisse üstünde test (artır = daha sağlam, daha yavaş)
)
print(f"Gün içi: {INTERVAL_STR} (~{BARS_PER_DAY} bar/gün) | "
      f"SURE={CFG['SURE']} bar ≈ {CFG['SURE']/BARS_PER_DAY:.1f} işlem günü")
print("Sinyal: 5 kanıtlı kural (birebir) · Çıkış: STOP=1ATR · +1.5R yarı+BE · +3R · 10 bar · risk %1")

In [ ]:
# === HÜCRE 3: VERİ (gün içi bar) ===
tv = TvDatafeed()   # temiz veri için: TvDatafeed(kullanici, sifre)

def evren_getir():
    from tradingview_screener import Query
    _, df = (Query().set_markets("turkey")
             .select("name","close","volume","average_volume_10d_calc")
             .limit(CFG["UNIVERSE_LIMIT"]).get_scanner_data())
    df = df.dropna(subset=["name"]).copy()
    df["likit_mTL"] = df["close"]*df["average_volume_10d_calc"]/1e6   # günlük likidite (doğru ölçek)
    df = df[(df["close"]>=CFG["MIN_FIYAT"]) & (df["likit_mTL"]>=CFG["MIN_LIKIT_MTL"])]
    df = df.sort_values("likit_mTL", ascending=False)                # en likitten başla
    s = list(df["name"].astype(str).unique())
    print(f"Evren: {len(s)} hisse (likiditeye göre sıralı)"); return s

_BAR_CACHE = {}
def bar_getir(s, n_bars=None):
    n_bars = n_bars or CFG["N_BARS_TARAMA"]
    key = (s, n_bars)
    if key in _BAR_CACHE: return _BAR_CACHE[key]
    try:
        df = tv.get_hist(s, exchange="BIST", interval=TV_INTERVAL, n_bars=n_bars)
        if df is None or len(df) < 260:   # 252-bar aralık göstergesi için asgari geçmiş
            _BAR_CACHE[key] = None; return None
        df = df.rename(columns=str.lower)[["open","high","low","close","volume"]]
        _BAR_CACHE[key] = df; return df
    except Exception:
        _BAR_CACHE[key] = None; return None

In [ ]:
# === HÜCRE 4: GÖSTERGELER (günlük backtest ile BİREBİR aynı — sadece bar zaman dilimi değişti) ===
def ema(s,n): return s.ewm(span=n,adjust=False).mean()
def atr(df,n=14):
    h,l,c=df["high"],df["low"],df["close"]; pc=c.shift()
    return pd.concat([h-l,(h-pc).abs(),(l-pc).abs()],axis=1).max(axis=1).ewm(alpha=1/n,adjust=False).mean()
def rsi(s,n=14):
    d=s.diff(); up=d.clip(lower=0).ewm(alpha=1/n,adjust=False).mean()
    dn=(-d.clip(upper=0)).ewm(alpha=1/n,adjust=False).mean()
    return 100-100/(1+up/dn.replace(0,np.nan))
def cmf(df,n=20):
    mfm=((df["close"]-df["low"])-(df["high"]-df["close"]))/(df["high"]-df["low"]).replace(0,np.nan)
    return (mfm*df["volume"]).rolling(n).sum()/df["volume"].rolling(n).sum()
def williams_r(df,n=14):
    hh=df["high"].rolling(n).max(); ll=df["low"].rolling(n).min()
    return -100*(hh-df["close"])/(hh-ll).replace(0,np.nan)
def bb_percent_b(s,n=20,k=2):
    m=s.rolling(n).mean(); sd=s.rolling(n).std(); return (s-(m-k*sd))/((2*k*sd).replace(0,np.nan))
def boga_engulfing(df):
    po,pc=df["open"].shift(),df["close"].shift()
    return (pc<po)&(df["close"]>df["open"])&(df["close"]>=po)&(df["open"]<=pc)
def _ramp(x,lo,hi): return ((x-lo)/(hi-lo)).clip(0,1)*100
def tech_cheapness(df):
    # 0-100 teknik ucuzluk (yüksek=daha dövülmüş) — backtest ile birebir ağırlıklar
    c=df["close"]; r=rsi(c,14)
    hi52=df["high"].rolling(252,min_periods=60).max(); lo52=df["low"].rolling(252,min_periods=60).min()
    pos=(c-lo52)/(hi52-lo52).replace(0,np.nan); dd=(hi52-c)/hi52.replace(0,np.nan)
    pctb=bb_percent_b(c); ema200=c.ewm(span=200,adjust=False).mean(); dev=(c-ema200)/ema200
    wr=williams_r(df,14); rslow=rsi(c,70)
    return (0.28*_ramp(pos,0.85,0.05)+0.20*_ramp(r,50,15)+0.12*_ramp(rslow,55,25)
            +0.15*_ramp(dd,0.10,0.60)+0.10*_ramp(pctb,0.50,-0.05)+0.10*_ramp(-dev,0.0,0.25)
            +0.05*_ramp(-wr,20,85))
def ichimoku_bull(df):
    h,l,c=df["high"],df["low"],df["close"]
    ten=(h.rolling(9).max()+l.rolling(9).min())/2; kij=(h.rolling(26).max()+l.rolling(26).min())/2
    a=((ten+kij)/2).shift(26); b=((h.rolling(52).max()+l.rolling(52).min())/2).shift(26)
    bulut=np.maximum(a,b); return (c>bulut)&(ten>kij)&(c.shift(1)<=bulut.shift(1))

In [ ]:
# === HÜCRE 5: KANITLI 5 SİNYAL — SERİ ÜRETEN (backtest + canlı ORTAK, birebir tanım) ===
# Not: Orijinal kod son bar için bool üretiyordu. Burada aynı koşulları TÜM geçmiş için
# seri olarak üretiyoruz (tanımlar birebir aynı) — hem canlı hem backtest aynı kodu kullanır.
def banker_seri(df):                                   # 🏆 günlükte +0.264R
    c=df["close"]
    return (tech_cheapness(df)>=CFG["GATE"]) & (cmf(df,20)>0.02) & (c.pct_change(20)<-0.02)
def derin_deger_seri(df):                              # +0.046R
    return tech_cheapness(df) >= CFG["GATE"]
def ichimoku_seri(df):                                 # +0.066R
    return ichimoku_bull(df)
def engulfing_seri(df):                                # +0.050R
    c=df["close"]; return boga_engulfing(df) & (c>ema(c,20))
def cmi_kombo_seri(df):                                # +0.033R
    c=df["close"]; k=cmf(df,20)
    return (k>0)&(k>k.shift(1))&boga_engulfing(df)&(df["volume"]>1.2*df["volume"].rolling(20).mean())

SINYAL_SERI = {"BANKER":banker_seri,"DERIN_DEGER":derin_deger_seri,"ICHIMOKU":ichimoku_seri,
               "ENGULFING":engulfing_seri,"CMI_KOMBO":cmi_kombo_seri}

def sinyalleri_hesapla(df):
    # son bar için 5 sinyalin durumu (bool) — canlı tarama için
    s = {k: bool(fn(df).fillna(False).iloc[-1]) for k,fn in SINYAL_SERI.items()}
    return s, float(tech_cheapness(df).iloc[-1])

# konviksiyon ağırlıkları — excess-R'ye orantılı (BANKER premium), günlük ile birebir
AGIRLIK = {"BANKER":3,"ICHIMOKU":1,"ENGULFING":1,"DERIN_DEGER":1,"CMI_KOMBO":1}
def konviksiyon(s):
    p=sum(AGIRLIK[k] for k,v in s.items() if v)
    if p>=4: return "GÜÇLÜ AL",p
    if p>=2: return "AL",p
    if p>=1: return "İZLE",p
    return "—",p

In [ ]:
# === HÜCRE 6: ÇIKIŞ SİMÜLASYONU — backtest + canlı defter ORTAK (bracket birebir) ===
# STOP=−1R · +1.5R'de yarı sat + stop'u başabaşa çek · +3R tam çık · SURE bar sonunda kapanışta çık.
def simule_cikis(gel_high, gel_low, gel_close, giris, stop, h1, h2, sure):
    # gel_* : giriş bar'ından SONRAKİ bar dizileri (numpy).
    R = giris - stop
    if R <= 0: return "GEÇERSİZ", np.nan
    n = int(min(sure, len(gel_close)))
    if n == 0: return "VERİ_YOK", np.nan
    yari=False; sc=stop
    for i in range(n):
        hi=gel_high[i]; lo=gel_low[i]
        if not yari:
            if lo<=sc: return "STOP", -1.0
            if hi>=h2: return "HEDEF2", 3.0
            if hi>=h1: yari=True; sc=giris            # yarı satıldı → stop başabaşa
        else:
            if lo<=sc: return "H1+BE", 0.75           # yarı +1.5R (0.75R) + kalan başabaş (0)
            if hi>=h2: return "HEDEF2", 2.25          # yarı +1.5R (0.75R) + kalan +3R (1.5R)
    # süre doldu → kapanışta işaretle
    son=gel_close[n-1]; rr=(son-giris)/R
    if yari: rr=0.75+0.5*rr
    return "SÜRE", float(rr)

In [ ]:
# === HÜCRE 7: SEVİYELER + POZİSYON + CANLI TARAMA (gün içi) ===
def degerlendir(sembol, df):
    if df is None or len(df)<200: return None
    s,tc=sinyalleri_hesapla(df); karar,puan=konviksiyon(s)
    if karar in ("—","İZLE"): return None            # sadece AL/GÜÇLÜ AL listeye
    c=df["close"]; giris=float(c.iloc[-1]); R=float(CFG["ATR_STOP"]*atr(df,CFG["ATR_LEN"]).iloc[-1])
    if R<=0: return None
    lot=int((CFG["SERMAYE"]*CFG["RISK_PCT"])/R); lot=min(lot,int(CFG["SERMAYE"]/giris))
    if lot<=0: return None
    v20=df["volume"].rolling(20).mean().iloc[-1]
    return {"Hisse":sembol,"Karar":karar,"Konviksiyon":puan,
            "Sinyaller":", ".join([k for k,v in s.items() if v]),
            "Ucuzluk":round(tc,0),"Giriş":round(giris,2),
            "Stop":round(giris-R,2),"Hedef1":round(giris+CFG["T1_R"]*R,2),
            "Hedef2":round(giris+CFG["T2_R"]*R,2),"Lot":lot,
            "Likit_mTL":round(float(v20*giris/1e6),1),"_df":df}

def tara():
    liste=[]; semboller=evren_getir()
    if CFG["TARAMA_SEMBOL"]: semboller=semboller[:CFG["TARAMA_SEMBOL"]]
    print(f"Taranacak: {len(semboller)} hisse ({INTERVAL_STR}, {CFG['N_BARS_TARAMA']} bar/hisse) "
          f"— tüm evren için biraz sürebilir...")
    for k,s in enumerate(semboller,1):
        if k%50==0: print(f"  {k}/{len(semboller)} · {len(liste)} sinyal")
        r=degerlendir(s, bar_getir(s))
        if r: liste.append(r)
    if not liste: print("\nŞu an AL sinyali yok — zorlama, nakit kal."); return pd.DataFrame()
    df=pd.DataFrame(liste)
    sira={"GÜÇLÜ AL":0,"AL":1}
    df=df.sort_values(["Karar","Konviksiyon","Likit_mTL"],
                      key=lambda c:c.map(sira) if c.name=="Karar" else c,
                      ascending=[True,False,False]).head(CFG["MAX_AL"]).reset_index(drop=True)
    df.insert(0,"Sıra",range(1,len(df)+1)); return df

sonuc=tara()
gor=[c for c in sonuc.columns if c!="_df"]
if not sonuc.empty:
    print(f"\n{'='*78}\n  ✅ GÜN İÇİ KANITLI AL LİSTESİ — {datetime.now():%Y-%m-%d %H:%M} · {INTERVAL_STR}\n{'='*78}")
    print(sonuc[gor].to_string(index=False))

## 🔬 Gün içi yeniden doğrulama — edge gerçekten taşındı mı?

Aşağıdaki hücre, **aynı 5 sinyali** ve **aynı çıkış bracket'ini** gün içi bar'larda tarihsel
olarak test eder ve **rastgele girişe** karşı `excess_R` (fazla getiri) ile `t` (istatistik)
üretir — orijinal günlük backtest'in mantığıyla aynı. Ayrıca canlı taramanın fiilen işlediği
**AL_KOMBO** (konviksiyon≥2) ve **GÜÇLÜ_AL** (≥4) kombinasyonlarını da ölçer.

**Nasıl okunur:** `excess_R > 0` ve `t ≳ 2` → o sinyalin edge'i bu zaml diliminde de var.
`excess_R ≤ 0` → o sinyale bu zaman diliminde **güvenme** (canlı işleme sokma).

> Bu, tek-hisse bracket'iyle dürüst bir *elek testidir* (portföy backtest'i değil): üst üste
> binen sinyaller işlem sayısını şişirebilir, komisyon/kayma dahil değildir. Amaç "edge gün
> içine taşındı mı?" sorusuna hızlı ve dürüst cevap vermek.

In [ ]:
# === HÜCRE 9: GÜN İÇİ YENİDEN DOĞRULAMA — sinyal vs rastgele giriş (excess-R, t) ===
rng = np.random.default_rng(42)

def backtest_sembol(df, cfg):
    a=atr(df,cfg["ATR_LEN"]).values
    H=df["high"].values; L=df["low"].values; C=df["close"].values; n=len(C); sure=cfg["SURE"]
    def _R_at(i):
        stop=C[i]-cfg["ATR_STOP"]*a[i]
        if not (np.isfinite(stop) and (C[i]-stop)>0): return None
        d=C[i]-stop; h1=C[i]+cfg["T1_R"]*d; h2=C[i]+cfg["T2_R"]*d
        _,rr=simule_cikis(H[i+1:i+1+sure],L[i+1:i+1+sure],C[i+1:i+1+sure],C[i],stop,h1,h2,sure)
        return rr
    def _topla(idx):
        idx=idx[(idx>=200)&(idx<=n-2)]
        rr=[_R_at(int(i)) for i in idx]
        return [r for r in rr if r is not None and np.isfinite(r)]
    # 5 sinyal serisi (bir kez hesapla)
    sig_arr={isim: fn(df).fillna(False).values.astype(bool) for isim,fn in SINYAL_SERI.items()}
    out={isim:_topla(np.where(sig)[0]) for isim,sig in sig_arr.items()}
    # konviksiyon kombinasyonları (canlı taramanın işlediği ürün)
    W=np.zeros(n)
    for isim,sig in sig_arr.items(): W += AGIRLIK[isim]*sig.astype(float)
    out["AL_KOMBO"]=_topla(np.where(W>=2)[0]); out["GÜÇLÜ_AL"]=_topla(np.where(W>=4)[0])
    # rastgele baseline (aynı çıkış, rastgele giriş)
    toplam=sum(len(out[k]) for k in SINYAL_SERI); pool=np.arange(200,n-1)
    k=min(len(pool), max(50, toplam))
    ridx=rng.choice(pool,size=k,replace=False) if len(pool)>0 else np.array([],int)
    out["_RASTGELE"]=_topla(np.asarray(ridx,int))
    return out

def _welch_t(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float)
    if len(a)<2 or len(b)<2: return np.nan
    va=a.var(ddof=1)/len(a); vb=b.var(ddof=1)/len(b)
    return (a.mean()-b.mean())/np.sqrt(va+vb) if (va+vb)>0 else np.nan

def backtest_calistir():
    semboller=evren_getir()[:CFG["BT_SEMBOL"]]
    anahtar=list(SINYAL_SERI)+["AL_KOMBO","GÜÇLÜ_AL","_RASTGELE"]
    biriktir={k:[] for k in anahtar}
    for j,s in enumerate(semboller,1):
        if j%20==0: print(f"  backtest {j}/{len(semboller)} hisse")
        df=bar_getir(s, CFG["N_BARS_BACKTEST"])
        if df is None or len(df)<300: continue
        r=backtest_sembol(df,CFG)
        for k,v in r.items(): biriktir[k]+=v
    return biriktir

print(f"Gün içi backtest başlıyor ({INTERVAL_STR}, en likit {CFG['BT_SEMBOL']} hisse)...")
biriktir=backtest_calistir()
base=np.array(biriktir["_RASTGELE"],float); base_mean=base.mean() if len(base) else 0.0
rows=[]
for k in list(SINYAL_SERI)+["AL_KOMBO","GÜÇLÜ_AL"]:
    v=np.array(biriktir[k],float)
    if len(v)==0: continue
    rows.append({"Sinyal":k,"islem":len(v),"ort_R":round(float(v.mean()),3),
                 "excess_R":round(float(v.mean()-base_mean),3),
                 "t_vs_rastgele":round(_welch_t(v,base),2),
                 "win%":round(float((v>0).mean()*100),1)})
bt=pd.DataFrame(rows).sort_values("excess_R",ascending=False).reset_index(drop=True)

print(f"\nRastgele giriş baseline: {len(base)} işlem · ort {base_mean:+.3f}R")
print(f"{'='*74}\n  GÜN İÇİ ({INTERVAL_STR}) YENİDEN DOĞRULAMA — sinyal vs rastgele giriş\n{'='*74}")
print(bt.to_string(index=False))
print("\nGünlük referans (bu koddan): BANKER +0.264R · ICHIMOKU +0.066R · ENGULFING +0.050R · "
      "DERIN_DEGER +0.046R · CMI_KOMBO +0.033R")
print("KURAL: excess_R>0 ve t≳2 → edge gün içinde de var. Aksi halde bu sinyali gün içinde İŞLEME SOKMA.")

In [ ]:
# === HÜCRE 10: BACKTEST GÖRSELİ (gün içi excess-R) ===
if not bt.empty:
    fig,ax=plt.subplots(figsize=(10,5))
    renk=["#3fb950" if x>0 else "#f85149" for x in bt["excess_R"]]
    ax.barh(bt["Sinyal"][::-1], bt["excess_R"][::-1], color=renk[::-1])
    ax.axvline(0,color="#8b949e",lw=1)
    ax.set_xlabel("excess_R (rastgele girişe göre fazla getiri, R cinsinden)")
    ax.set_title(f"Gün içi ({INTERVAL_STR}) yeniden doğrulama — sinyal edge'i")
    for i,(sig,x,t) in enumerate(zip(bt["Sinyal"][::-1],bt["excess_R"][::-1],bt["t_vs_rastgele"][::-1])):
        ax.text(x, i, f"  {x:+.3f}R (t={t})", va="center",
                ha="left" if x>=0 else "right", fontsize=9)
    ax.grid(alpha=.3, axis="x"); plt.tight_layout(); plt.show()

### 🔎 Gün içi doğrulama bulguları — 1h koşusu (80 hisse, ~2.5 yıl)

> Aşağıdaki tablo **fiilen çalıştırılan 1h backtest'in sonucudur** (Hücre 9). Kayıt için
> buraya alındı — strateji kodu bilinçli olarak **değiştirilmedi**; bu bir gözlem notudur.
> Kendi koşunda sayılar biraz oynayabilir, ama örüntü büyük ihtimalle aynı kalır.

| Sinyal | işlem | ort_R | **excess_R** | **t** | Karar (kural: excess_R>0 & t≳2) |
|---|---|---|---|---|---|
| **ENGULFING** | 13.467 | 0.203 | **+0.050** | **3.51** | ✅ **GEÇTİ** — günlükle (+0.050R) neredeyse birebir |
| AL_KOMBO | 639 | 0.262 | +0.109 | 1.88 | 🟡 en yüksek fazlalık ama t<2 (sınırda) |
| ICHIMOKU | 4.482 | 0.173 | +0.019 | 0.86 | ❌ anlamsız (günlük +0.066R'den zayıfladı) |
| DERIN_DEGER | 21.425 | 0.080 | **−0.073** | **−6.11** | ⛔ **ZARARLI** — günlük +0.046R gün içinde tersine döndü |
| BANKER · CMI_KOMBO · GÜÇLÜ_AL | 0 | — | — | — | ⚫ **hiç tetiklenmedi** (bkz. aşağıda) |

**Rastgele giriş baseline: +0.153R.** Kritik satır: 1h BIST'te *rastgele* bir giriş bile bu
çıkış bracket'iyle +0.153R veriyor (TL bazlı enflasyon sürüklenmesi). Bu yüzden `ort_R` sütunu
yanıltıcıdır — herkes pozitif görünür; **gerçek edge = rastgeleye göre `excess_R`.** Günlük
referansların da küçük olması (+0.046R…) bu metrikle tutarlı.

#### Üç ana çıkarım
1. **BANKER gün içinde YOK (0 işlem).** Günlük #1 şampiyonun üç koşulu (`tech_cheapness≥60` &
   `CMF>0.02` & `20-bar düşüş`) 1h'te neredeyse hiç çakışmıyor: saatlik bazda hisse "ucuz/
   düşüşte" iken CMF genelde negatif. BANKER olmayınca GÜÇLÜ_AL de yok → **BANKER-ağırlıklı
   konviksiyon şeması (BANKER=3) gün içinde işlevsiz.**
2. **DERIN_DEGER gün içinde zararlı (t=−6.11).** "Teknik ucuz" saatlik bazda *düşen bıçağı
   yakalamak* demek; dönüş için zaman yok. Günlük edge tersine döndü — timeframe'i körü körüne
   değiştirmenin klasik tuzağı. Canlıda **standalone DERIN_DEGER girişi açma.**
3. **ENGULFING tek sağlam kazanan** (t=3.51). Kısa-vadeli dönüş mumu olduğu için timeframe'e
   dayanıklı; excess_R'si günlükle neredeyse aynı. **AL_KOMBO** (+0.109R) umut verici — pratikte
   "ENGULFING + en az bir teyit" — ama t=1.88, daha çok işlemle doğrulanmalı.

#### Sonuç (henüz uygulanmadı — kullanıcı kararına bırakıldı)
Mevcut canlı tarayıcı (Hücre 7) AL_KOMBO/GÜÇLÜ_AL konviksiyonunu işler; bu şema zararlı
DERIN_DEGER'i içerir ve var olmayan BANKER'a dayanır. **1h'te canlıya sokmadan önce** ya (a)
tarayıcıyı ENGULFING çekirdekli + DERIN_DEGER standalone çıkarılmış biçimde yeniden kur, ya da
(b) 15m/30m'de aynı backtest'i çalıştırıp edge'in orada güçlenip güçlenmediğine bak. Bu karar
verilene kadar strateji kodu olduğu gibi bırakıldı.

In [ ]:
# === HÜCRE 11: KAYDET + İŞLEM DEFTERİ (gün içi zaman damgalı) ===
DEFTER=os.path.join(BASE,f"islem_defteri_{INTERVAL_STR}.csv")
if not sonuc.empty:
    st=datetime.now().strftime("%Y%m%d_%H%M")
    xls=os.path.join(BASE,f"kanitli_al_{INTERVAL_STR}_{st}.xlsx"); sonuc[gor].to_excel(xls,index=False)
    print("💾",xls)
    kayit=sonuc[gor].copy(); kayit.insert(0,"tarama_zamani",datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    kayit.insert(1,"interval",INTERVAL_STR); kayit["sonuc"]=""; kayit["R_sonuc"]=np.nan
    if os.path.exists(DEFTER): kayit=pd.concat([pd.read_csv(DEFTER),kayit],ignore_index=True)
    kayit.to_csv(DEFTER,index=False); print("📓 Defter:",DEFTER,f"({len(kayit)} kayıt)")

In [ ]:
# === HÜCRE 12: AÇIKLAMALI GRAFİK (giriş/stop/hedef) ===
def grafik(row,bar=160):
    df=row["_df"].tail(bar); c=df["close"]
    e20=ema(row["_df"]["close"],20).tail(bar); e50=ema(row["_df"]["close"],50).tail(bar)
    fig,ax=plt.subplots(figsize=(12,6))
    ax.plot(c.index,c,color="#58a6ff",lw=1.5,label="Kapanış")
    ax.plot(e20.index,e20,color="#f0883e",lw=1,label="EMA20"); ax.plot(e50.index,e50,color="#a371f7",lw=1,label="EMA50")
    for sev,rk,et in [(row["Giriş"],"#e6edf3","GİRİŞ"),(row["Stop"],"#f85149","STOP −1R"),
                      (row["Hedef1"],"#3fb950","H1 +1.5R"),(row["Hedef2"],"#2ea043","H2 +3R")]:
        ax.axhline(sev,color=rk,ls="--",lw=1,alpha=.8); ax.text(c.index[-1],sev,f" {et} {sev:.2f}",color=rk,va="center",fontsize=9)
    ax.set_title(f"{row['Hisse']} · {row['Karar']} ({row['Sinyaller']}) · Lot {row['Lot']} · {INTERVAL_STR}",color="#e6edf3")
    ax.legend(facecolor="#161b22",edgecolor="#30363d",labelcolor="#e6edf3",loc="upper left"); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()
if not sonuc.empty:
    for _,row in sonuc.head(5).iterrows(): grafik(row)

In [ ]:
# === HÜCRE 13: İŞLEM DEFTERİNİ DEĞERLENDİR (canlı R — backtest'le kıyas) ===
# Çıkış Hücre 6'daki simule_cikis ile birebir aynı — canlı & backtest tutarlı.
def defteri_degerlendir():
    if not os.path.exists(DEFTER): print("Defter yok."); return
    d=pd.read_csv(DEFTER); acik=d[d["R_sonuc"].isna()]
    if acik.empty: print("Açık kayıt yok."); return
    for idx,row in acik.iterrows():
        df=bar_getir(row["Hisse"])
        if df is None: continue
        ts=pd.Timestamp(row["tarama_zamani"])
        ile=df[df.index>ts]
        if len(ile)<1: continue
        sonuc,rr=simule_cikis(ile["high"].values,ile["low"].values,ile["close"].values,
                              row["Giriş"],row["Stop"],row["Hedef1"],row["Hedef2"],CFG["SURE"])
        d.loc[idx,"sonuc"]=sonuc; d.loc[idx,"R_sonuc"]=round(rr,3) if rr==rr else np.nan
    d.to_csv(DEFTER,index=False)
    k=d[d["R_sonuc"].notna()]
    if len(k):
        print(f"Kapanan: {len(k)} · ort {k['R_sonuc'].mean():+.3f}R · win %{(k['R_sonuc']>0).mean()*100:.0f}")
        for kr in ["GÜÇLÜ AL","AL"]:
            kk=k[k["Karar"]==kr]
            if len(kk): print(f"  {kr}: {len(kk)} işlem · ort {kk['R_sonuc'].mean():+.3f}R")
        print("(Karşılaştır: Hücre 9 gün içi backtest sonuçların ile — canlı ondan sapmamalı.)")
    return d
print("Birkaç bar/gün sonra çalıştır:  defter = defteri_degerlendir()")

## 📖 Gün içi nasıl çalışmalıyım

### Ritim (zaman dilimine göre)
1. **Her yeni bar kapanışında** (örn. 1h'te saat başı) Hücre 1→7'yi çalıştır → gün içi AL listesi.
   Sinyal **kapanan bar** üzerinde üretilir; girişi **bir sonraki bar açılışında** yap (look-ahead yok).
2. Emir kurarken **STOP + HEDEF**'i hemen gir. Konviksiyona göre önceliklendir:
   **GÜÇLÜ AL (BANKER içeren) > AL**.
3. Hücre 11 defteri güncelle; birkaç bar sonra Hücre 13 ile canlı R'yi ölç.

### Pozisyon yönetimi (backtest bracket'i — DEĞİŞTİRME)
- **Lot** = %1 riske göre; listedekini kullan.
- **HEDEF1 (+1.5R):** yarı sat + stop'u girişe (başabaş) çek → risksiz koşu.
- **HEDEF2 (+3R):** kalanı sat.  · **STOP (−1R):** değerse çık, tartışma yok.
- **SURE = 10 bar:** dolduysa kapanışta çık. (1h'te ≈ 1.25 işlem günü; 15m'de ≈ 2.5 saat.)

### Gün içine özgü uyarılar
- **Önce Hücre 9'u çalıştır.** Sadece `excess_R>0 & t≳2` çıkan sinyalleri canlıda işle.
  Örn. BANKER günlükte açık ara #1'di; gün içinde de öyle mi — tabloya bak, körü körüne varsayma.
- **Likidite kritik.** Gün içinde ince hacimli hisselerde kayma stopu yer. `TARAMA_SEMBOL`
  en likit hisselerle sınırlı; düşürme.
- **Zaman dilimini değiştirdiysen** (5m/15m/2h) **Hücre 9'u yeniden çalıştır** — edge her
  zaml diliminde aynı değildir.
- **SURE'yi bar sanma tuzağı:** `SURE=10` gün değil **bar**. Zaman dilimine göre gerçek süreyi
  Hücre 2 çıktısı yazıyor.

### Değişmeyen ilkeler
- Kazanan tema **birikim + dönüş**, kırılım DEĞİL. Elenen kuralları (kırılım/momentum/MACD…)
  geri ekleme. yfinance kullanma. Sinyal yoksa işlem uydurma.
- Edge küçük; **50-100 işlemde** ortaya çıkar. Tek partiden karar verme, disiplini koru.

> **Yatırım tavsiyesi değildir.** Karar ve risk kullanıcıya aittir.